<a href="https://colab.research.google.com/github/svyatoslavna/nlp_hw/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Тонкая настройка моделей в Hugging Face (fine-tuning)

На входе у нас всегда **pre-trained** модель:
BERT, GPT-2, BART, Pegasus...

Они обучены на:
1. (часто) многоязычных данных
2. данные общей предметной области
3. обобщающая задача:
    - Natural Language Inference
        - Классификация
        - 1) Мопс лежал на диване
        - 2) Мопс скакал // Мопс был бежевым // Он посапывал
        - Модель выбирает одну из 3х меток: **entailment**, **contradiction**, **neutral**
    - Masked Language Modeling
        - Cloze test
        - Я пошел сегодня в [MASK]. Купил хлеб.
        - Ответ модели: магазин (0.4), булочную (0.3), киоск (0.1)...
    - Next Sentence Prediction
    - Language Modeling

Типы файн-тюнининга:
- Domain Adaptation: open domain -> medical domain; medical domain -> pharma
- Language Transfer: rich resource -> low resource
- Task Transfer: BERT + линейный слой
- Multilingual Learning: пре-трейн на множестве языков
- Multitask Learning: T5

In [3]:
# Установка библиотек
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

# Для работы с GPU (проверяем наличие)
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


#### Фаза 1: Fine-tune на датасете IMDB

In [4]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
# 1. Загрузка датасета
dataset = load_dataset("stanfordnlp/imdb")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Датасет загружен. Train: 25000, Test: 25000


In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [21]:
dataset['test'][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

In [22]:
# 2. Загрузка модели и токенизатора
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # 0 = негативный, 1 = позитивный
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
# 3. Подготовка данных
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Здесь мы используем `map` с `batched=True`, чтобы датасет срез датасета через токенизатор. `.select(range(5000))` берёт 5000 образцов

In [24]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir="./results-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

**TensorBoard** — инструмент визуализации от TensorFlow. Позволяет отслеживать в реальном времени:
- кривые обучения (loss, accuracy)
- распределения весов
- эмбеддинги
- график модели

В `Trainer` из `transformers` он используется по умолчанию для логирования метрик. Если тензорборд не нужен, достаточно установить `report_to="none"`

In [25]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [26]:
# 6. Обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.258531,0.252411,0.897000
2,0.165317,0.276764,0.907800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.2309666924650518, metrics={'train_runtime': 1286.2214, 'train_samples_per_second': 38.874, 'train_steps_per_second': 2.43, 'total_flos': 3311668407975168.0, 'train_loss': 0.2309666924650518, 'epoch': 2.0})

In [27]:
# 7. Оценка
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")

Training Loss,Validation Loss,Epoch,Accuracy
0.165317,0.276764,2,0.907800



Evaluation results: {'eval_loss': 0.27676430344581604, 'eval_accuracy': 0.9078}


In [28]:
# 8. Инференс через pipeline
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "This movie was fantastic! I loved every minute.",
    "A complete waste of time. Terrible acting and plot."
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

Text: This movie was fantastic! I loved every minute.
Sentiment: LABEL_1, Score: 0.9963

Text: A complete waste of time. Terrible acting and plot.
Sentiment: LABEL_0, Score: 0.9965



In [29]:
# 9. Демо-интерфейс с Gradio
import gradio as gr

def predict_sentiment(text):
    result = classifier(text)[0]
    return {result['label']: result['score']}

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="Введите отзыв на фильм"),
    outputs=gr.Label(label="Тональность"),
    title="Анализ тональности отзывов (IMDB)",
    examples=[
        ["This movie was fantastic! I loved every minute."],
        ["A complete waste of time. Terrible acting and plot."],
        ["Pretty good film, worth watching."]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdcd819889934db550.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Итог**

Мы выполнили fine-tuning модели `distilbert-base-uncased` на датасете `stanfordnlp/imdb` для задачи определения тональности отзыва на фильм. **Accuracy на тестовой выборке достигла 90%**, что является отличным результатом. Инференс на примерах отзывов каждого класса — negative, positive — был выполнен корректно.

#### Фаза 2: Fine-tune на датасете `rotten_tomatoes` с моделью BERT

Меняем задачу (классификация отзывов на фильмы) и модель (`bert-base-uncased`)

In [5]:
# 1. Загрузка нового датасета cornell-movie-review-data/rotten_tomatoes
dataset_rt = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(dataset_rt)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [6]:
# 2. Загрузка другой модели и токенизатора
model_name_bert = "google-bert/bert-base-uncased" # допишите код для загрузки bert-base-uncased
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert) # допишите код для загрузки токенизатора
model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name_bert,
    num_labels=2
).to(device) # допишите код для загрузки модели для классификации

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
lengths = [len(tokenizer_bert(text)["input_ids"]) for text in dataset_rt["train"]["text"]]
print(f"Средняя длина: {np.mean(lengths):.0f}")
print(f"95 персентиль: {np.percentile(lengths, 95)}")

Средняя длина: 27
95 персентиль: 47.0


In [8]:
# 3. Подготовка данных
def tokenize_rt(examples):
    return tokenizer_bert(examples["text"], truncation=True, max_length=50)

tokenized_rt = dataset_rt.map(tokenize_rt, batched=True)
train_rt = tokenized_rt["train"]
test_rt = tokenized_rt["test"]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [9]:
# 4. Настройка обучения
training_args_rt = TrainingArguments(
    output_dir="./rt_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [11]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [12]:
# 6. Обучение

trainer_rt = Trainer(
    model=model_bert,
    args=training_args_rt,
    train_dataset=train_rt,
    eval_dataset=test_rt,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer_bert),
)

print("Training on Rotten Tomatoes...")
trainer_rt.train()

Training on Rotten Tomatoes...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.391958,0.389919,0.833021
2,0.228344,0.458682,0.851782
3,0.126668,0.625157,0.851782


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1602, training_loss=0.23933764373169708, metrics={'train_runtime': 300.8354, 'train_samples_per_second': 85.063, 'train_steps_per_second': 5.325, 'total_flos': 612876092358720.0, 'train_loss': 0.23933764373169708, 'epoch': 3.0})

In [13]:
# 7. Оценка
rt_results = trainer_rt.evaluate() # получите метрики оценки
print(f"Rotten Tomatoes results: {rt_results}")

Training Loss,Validation Loss,Epoch,Accuracy
0.126668,0.459625,3,0.852720


Rotten Tomatoes results: {'eval_loss': 0.45962512493133545, 'eval_accuracy': 0.8527204502814258}


In [16]:
# 8. Инференс
from transformers import pipeline

classifier_rt = pipeline("sentiment-analysis", model=model_bert, tokenizer=tokenizer_bert) # загрузите пайплайн для анализа тональности

movie_reviews = [
    "A masterpiece of modern cinema. The direction is flawless.",
    "Boring and predictable. I fell asleep halfway through."
]

for review in movie_reviews:
    res = classifier_rt(review)[0]
    print(f"Review: {review}\nVerdict: {res['label']}, Confidence: {res['score']:.4f}\n")

Review: A masterpiece of modern cinema. The direction is flawless.
Verdict: LABEL_1, Confidence: 0.9970

Review: Boring and predictable. I fell asleep halfway through.
Verdict: LABEL_0, Confidence: 0.9939



**Итог**

Мы выполнили fine-tuning модели `google-bert/bert-base-uncased` на датасете `cornell-movie-review-data/rotten_tomatoes` для задачи определения тональности отзыва на фильм. **Accuracy на тестовой выборке достигла 85%**, что является хорошим результатом. Инференс на примерах отзывов каждого класса — negative, positive — был выполнен корректно.

In [ ]:
# 10. Сохранение модели

model_bert.save_pretrained('./my_model')
tokenizer_bert.save_pretrained('./my_model')

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...vnersjr/model.safetensors:   0%|          |  574kB /  268MB            

CommitInfo(commit_url='https://huggingface.co/missvector/bert-sentiment-imdb/commit/839a5ff20165faabec446246571ad611a3e75126', commit_message='Upload DistilBertForSequenceClassification', commit_description='', oid='839a5ff20165faabec446246571ad611a3e75126', pr_url=None, repo_url=RepoUrl('https://huggingface.co/missvector/bert-sentiment-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='missvector/bert-sentiment-imdb'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r ./my_model '/content/drive/MyDrive/'